# Premier smile, hors ligne

Ce notebook lit `data/dev/marches.parquet`, tire par `scripts/pull_dev.py`. **Aucune connexion
a la base** : c'est tout l'interet du montage — le reseau ne fait que remplir le dossier, le
travail se fait en local, et ce qui est affiche ici est reproductible depuis le manifeste
(qui contient la requete exacte et le commit).

Si le fichier manque : `scripts/db-tunnel.sh start` puis `scripts/pull_dev.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

marches = pd.read_parquet("data/dev/marches.parquet")
perp = pd.read_parquet("data/dev/perpetuel.parquet")

print(f"{len(marches):,} lignes".replace(",", " "))
print("du", marches.observed_at.min(), "au", marches.observed_at.max())
print("instantanes :", marches.observed_at.nunique(), "| instruments :", marches.instrument_name.nunique())
print("expirations  :", marches.expiry.nunique(), "| sous-jacents :", sorted(marches.underlying.unique()))
print(f"IV presente  : {marches.mark_iv.notna().mean():.1%} | bid/ask present : {marches.bid_iv.notna().mean():.1%}")

In [ ]:
# Un seul instantane, une seule echeance, et le forward pris au meme instant.
instant = marches.observed_at.max()
fwd = (perp.loc[perp.observed_at <= instant]
           .sort_values("observed_at")
           .groupby("instrument_name").index_price.last())

for sous_jacent in sorted(marches.underlying.unique()):
    f = fwd.get(f"{sous_jacent}-PERPETUAL", np.nan)
    dispo = marches[(marches.observed_at == instant) & (marches.underlying == sous_jacent)]
    if dispo.empty or not np.isfinite(f):
        continue
    echeance = (dispo.groupby("expiry").size().loc[lambda s: s >= 15].index.min()
                if (dispo.groupby("expiry").size() >= 15).any() else dispo.expiry.min())
    tranche = dispo[dispo.expiry == echeance].sort_values("strike")
    k = np.log(tranche.strike / f)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(k, tranche.mark_iv, "o-", ms=3, lw=1, label="IV de marque")
    if tranche.bid_iv.notna().any():
        ax.fill_between(k, tranche.bid_iv, tranche.ask_iv, alpha=0.25, label="bid / ask")
    ax.axvline(0, color="0.6", lw=0.8, ls=":")
    ax.set_xlabel("log(K/F)  —  F = %.0f" % f)
    ax.set_ylabel("volatilite implicite (%)")
    ax.set_title(f"{sous_jacent} — echeance {echeance} — {instant}")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"data/dev/smile_{sous_jacent}.png", dpi=130)
    plt.show()
    print(f"{sous_jacent} : {len(tranche)} points, K de {tranche.strike.min():.0f} a {tranche.strike.max():.0f}")